##### Copyright 2026 The colab Authors.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
[apache.org/licenses/LICENSE-2.0](https://www.apache.org/licenses/LICENSE-2.0).

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS, WITHOUT
WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.

<table align="left"><td>
  <a target="_blank" href="https://colab.research.google.com/github/xbill9/colab/blob/main/notebooks/llamacpp_tpu_gemma4_e2b_v5e1.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab
  </a>
</td><td>
  <a target="_blank" href="https://github.com/xbill9/colab/blob/main/notebooks/llamacpp_tpu_gemma4_e2b_v5e1.ipynb">
    <img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub
  </a>
</td></table>

# llama.cpp on a Colab TPU: Gemma 4 E2B in three cells

**Author:** William McLean, Google Developer Expert

This notebook runs an unmodified llama.cpp `llama-server` on one Colab TPU v5e chip, through
[`ggml-tpu`](https://github.com/xbill9/ggml-tpu), an unofficial ggml backend that lowers each llama.cpp
graph to one XLA program. It serves Gemma 4 E2B behind an OpenAI-compatible endpoint and opens a chat
page in the notebook.

**Unofficial.** `ggml-tpu` is an independent project, not part of ggml-org/llama.cpp and not made or
endorsed by Google.

### What runs

* **llama.cpp** `llama-server` at commit `7182ef71e`, and **`libggml-tpu.so`**, from one prebuilt
  7 MB tarball ([release v0.1.0](https://github.com/xbill9/ggml-tpu/releases/tag/v0.1.0)). Nothing
  compiles C++ here.
* **The model:** [`xbill9/gemma-4-E2B-it-qat-q4_0-exact-gguf`](https://huggingface.co/xbill9/gemma-4-E2B-it-qat-q4_0-exact-gguf),
  Gemma 4 E2B with Google's quantization-aware-trained 4-bit weights stored exactly as Q4_0, 2.62 GB.
  Public and ungated.
* **The TPU runtime:** `libtpu`, the library JAX uses, loaded directly by the backend. This notebook
  never imports JAX: only one process can hold the TPU, and it has to be `llama-server`.

### Before you start

* **Runtime.** Runtime, then Change runtime type, then **v5e-1 TPU**. Then Runtime, Run all.
* **No token.** Every download is public.
* **The first reply waits for XLA.** Each new prompt length and every 256 tokens of context compile
  one program, a few seconds each, once per runtime. After that, decoding on a Cloud TPU v5e ran at
  109 to 112 tokens per second (2026-11-19).

## 1. Download

The model and the prebuilt tarball download in the background; this cell returns at once and reports
the TPU runtime library while they arrive.

In [ ]:
# Cell 1: start the downloads, find libtpu
import glob
import hashlib
import importlib.metadata as md
import importlib.util
import json
import os
import pathlib
import subprocess
import sys
import tarfile
import time
import urllib.request

T_START = time.perf_counter()
WORK = pathlib.Path("/content" if os.path.isdir("/content") else os.path.expanduser("~/llamacpp-tpu"))
WORK.mkdir(parents=True, exist_ok=True)

MODEL_REPO = "xbill9/gemma-4-E2B-it-qat-q4_0-exact-gguf"
MODEL_FILE = "gemma-4-E2B-it-q4_0-exact.gguf"
MODEL_REV = "74b2c92c8f7f84d3d9c39e8026f6a28f62788255"
RELEASE = "ggml-tpu-a4a425d-llama.cpp-7182ef71e-linux-x86_64"
RELEASE_URL = f"https://github.com/xbill9/ggml-tpu/releases/download/v0.1.0/{RELEASE}.tar.gz"
RELEASE_SHA256 = "400e2a629c28a5f262a185de09458b56ab0c19147bcd3232a4ab3aa81eea1beb"

DOWNLOAD = """
import sys
from huggingface_hub import hf_hub_download
print(hf_hub_download(sys.argv[1], sys.argv[2], revision=sys.argv[3], token=False))
"""
model_dl = subprocess.Popen(
    [sys.executable, "-c", DOWNLOAD, MODEL_REPO, MODEL_FILE, MODEL_REV],
    stdout=subprocess.PIPE,
    stderr=open(WORK / "download.log", "w"),
    text=True,
)

tarball = WORK / f"{RELEASE}.tar.gz"
urllib.request.urlretrieve(RELEASE_URL, tarball)
if hashlib.sha256(tarball.read_bytes()).hexdigest() != RELEASE_SHA256:
    raise RuntimeError(f"{tarball.name}: SHA-256 does not match the release")
with tarfile.open(tarball) as tf:
    tf.extractall(WORK)
BIN = WORK / RELEASE
print((BIN / "VERSION").read_text())

# libtpu ships with JAX on Colab TPU runtimes; install it only if it is missing
if importlib.util.find_spec("libtpu") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "libtpu"], check=True)
LIBTPU = next(
    iter(
        glob.glob(
            os.path.join(os.path.dirname(importlib.util.find_spec("libtpu").origin), "**", "libtpu.so"), recursive=True
        )
    ),
    None,
)
if LIBTPU is None:
    raise RuntimeError("libtpu.so not found: is this a TPU runtime? Runtime > Change runtime type > v5e-1 TPU")
for pkg in ("libtpu", "jax"):
    try:
        print(f"{pkg:7s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:7s} not installed")
print(f"libtpu.so: {LIBTPU}")
print(f"\ncell 1 done {time.perf_counter() - T_START:.0f} s after it started; the model is still downloading")

## 2. Serve

This cell waits for the model, starts `llama-server` on the TPU (device `TPU0`), then the chat page.
`llama-server` is ready once the weights are on the chip; the first reply then waits for XLA to
compile.

In [ ]:
# Cell 2: llama-server on the TPU, then the chat page
out, _ = model_dl.communicate()
if model_dl.returncode != 0:
    raise RuntimeError(f"model download failed:\n{(WORK / 'download.log').read_text()[-2000:]}")
MODEL_PATH = out.strip().splitlines()[-1]
print(f"model ready {time.perf_counter() - T_START:.0f} s after cell 1 started: {MODEL_PATH}")

PORT = 8080
env = dict(
    os.environ,
    GGML_BACKEND_PATH=str(BIN / "libggml-tpu.so"),
    GGML_TPU_LIBTPU=LIBTPU,
    GGML_TPU_PRECISION="DEFAULT",
)
server = subprocess.Popen(
    [
        str(BIN / "llama-server"),
        "-m",
        MODEL_PATH,
        "-dev",
        "TPU0",
        "-ngl",
        "99",
        "-fa",
        "off",
        "-c",
        "8192",
        "--host",
        "127.0.0.1",
        "--port",
        str(PORT),
        "--parallel",
        "1",
        "--metrics",
        "--jinja",
        "--alias",
        "gemma-4-E2B-it-q4_0-exact",
    ],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=open(WORK / "llama-server.log", "w"),
)
t0 = time.perf_counter()
while True:
    try:
        if urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2).status == 200:
            break
    except OSError:
        pass
    if server.poll() is not None:
        raise RuntimeError(f"llama-server exited:\n{(WORK / 'llama-server.log').read_text()[-3000:]}")
    time.sleep(0.2)
print(
    f"llama-server ready in {time.perf_counter() - t0:.1f} s, {time.perf_counter() - T_START:.0f} s after cell 1 started"
)

UI_PORT = 8765
ui = subprocess.Popen(
    [
        sys.executable,
        str(BIN / "chat_ui.py"),
        "--upstream",
        f"http://127.0.0.1:{PORT}",
        "--port",
        str(UI_PORT),
        "--rig",
        "ACCELERATOR_TYPE=Colab TPU v5e-1",
        "GGML_TPU_EXECUTOR=pjrt",
        "GGML_TPU_QUANT=bf16",
        "GGML_TPU_PRECISION=DEFAULT",
    ],
    stdout=subprocess.DEVNULL,
    stderr=open(WORK / "chat_ui.log", "w"),
)
for _ in range(50):
    try:
        json.load(urllib.request.urlopen(f"http://127.0.0.1:{UI_PORT}/ui/status", timeout=2))
        break
    except OSError:
        time.sleep(0.2)
else:
    raise RuntimeError(f"chat page did not start:\n{(WORK / 'chat_ui.log').read_text()[-2000:]}")

try:
    from google.colab import output

    output.serve_kernel_port_as_iframe(UI_PORT, height=720)
except ImportError:
    print(f"not on Colab: open http://127.0.0.1:{UI_PORT} in a browser on this machine")

## 3. Use it

Type into the chat page above. Its side panel turns thinking on and off and shows each reply's speed,
measured in the browser and by `llama-server`. From code, any OpenAI client works against
`http://127.0.0.1:8080/v1` inside this runtime.

When you are done, disconnect the runtime (Runtime, then Disconnect and delete runtime) so it stops
using compute units.

In [ ]:
# Cell 3: one request through the OpenAI-compatible endpoint
body = {
    "messages": [{"role": "user", "content": "Why is the sky blue? One sentence."}],
    "max_tokens": 128,
    "temperature": 0,
    "chat_template_kwargs": {"enable_thinking": False},
}
req = urllib.request.Request(
    f"http://127.0.0.1:{PORT}/v1/chat/completions", json.dumps(body).encode(), {"Content-Type": "application/json"}
)
t0 = time.perf_counter()
reply = json.load(urllib.request.urlopen(req, timeout=600))
print(reply["choices"][0]["message"]["content"])
timings = reply["timings"]
print(
    f"\n{timings['predicted_n']} tokens in {time.perf_counter() - t0:.1f} s; the server decoded at {timings['predicted_per_second']:.1f} tokens/s"
)
print("the first request at a new length includes an XLA compile; run this cell again to see the compiled speed")